In [ ]:
from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import numpy as np 
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.interpolate import interp1d
from sklearn.preprocessing import StandardScaler
import pandas as pd
import random

plt.rcParams.update({'font.size': 20})

def SM90(t,system,R,n=0):
    """
    Saltzman-Maasch 1990 model (SM90) from "A first-order global model of late Cenozoic climatic change";
    Late Pleistocene solution; 
    pages 320-321 

    t      - time 
    system - state vector
    R      - insolation forcing 
    n      - statistical forcing (e.g. normally distributed noise) 
    """

    X_t = system[0]
    Y_t = system[1]
    Z_t = system[2]
    R_t = R(t_cutoff-t)

    # random variable for noise inclusion if noise is non-yero
    b = random.choice([True, False])

    # prefactors for Late Pleistocene solution of the SM90 model 
    v = 0.2
    u = 0.6
    p = 1.0
    r = 0.9
    s = 1.0
    w = 0.5
    q = 2.5
    
    dX_dt = - X_t - Y_t - v*Z_t - u*R_t + b*n 
    dY_dt = - p*Z_t + r*Y_t + s*Z_t**2 - w*Y_t*Z_t - Z_t**2*Y_t + b*n 
    dZ_dt = - q*(X_t + Z_t) + b*n 

    return np.array([dX_dt, dY_dt, dZ_dt])

# time range for the numerical integration dependent on the used insolation data  
t_cutoff = 400 # in 10 kyr -- originally 400
dt = 0.1 # in 10 kyr (0.01*10,000 = 100yr or 0.1kyr)

# 0->95 in 0.1 increment, gives ~950 points
# 75% is 712 points, 50% is 475 points, 25% is 237 points
# 0->400 in 0.1 increment, give ~4000 points

color_dict = {"I":"coral","X":"lightskyblue","Y":"slategrey","Z":"limegreen"}

# insolation forcing, different to insolation used by Saltzman
R_Df = pd.read_csv(f"{PROJECT}/Data/Observations/huybers06_65north.csv",
                   comment="#",delimiter=" ",usecols=[0,1],names=["t","I"],skiprows=1)
R_Df = R_Df[R_Df["t"]<=t_cutoff*10]

# scale down time range according to \lambda_0 = 10 kyr used in Saltzman
R_Df["t"] *= 1e-1
R_Df = R_Df[::-1]

# scaling insolation to mean of 0 and variance of 1 in accordance with SM90 
R_Scaler = StandardScaler().fit(R_Df["I"].values.reshape(-1, 1))
R_Df["I"] = R_Scaler.transform(R_Df["I"].values.reshape(-1, 1))

# interpolate forcing since solve_ivp uses adaptable time step
R_forcing = interp1d(
    R_Df["t"], 
    R_Df["I"], 
    kind='linear',            
    bounds_error=False,        
    fill_value=0.0          
) 

# simulation time range
t = np.arange(0,t_cutoff,dt)
# initialiye state vector
system = np.zeros((len(t),3))

# set initial conditions 
system_0 = np.array([-1.0,0,1])

# numerical solver
sol = solve_ivp(SM90,t_span=[0,t_cutoff],y0=system_0,t_eval=t,args=(R_forcing,0))

fig, axs = plt.subplots(4,1,figsize=(18,24))

print(sol.t.shape)

axs[0].plot(sol.t,R_forcing(t),color=color_dict["I"])
#axs[0].axvline(156,linestyle="--",color="k",alpha=0.5)
#axs[0].axvline(156+50,linestyle="--",color="k",alpha=0.5)
axs[0].set_ylabel("I")
axs[0].xaxis.set_inverted(True) 
axs[0].set_xticks([])

axs[1].plot(sol.t,sol.y[0,:],color=color_dict["X"])
axs[1].axvline(156,linestyle="--",color="k",alpha=0.5)
axs[1].axvline(264.2+50,linestyle="--",color="k",alpha=0.5)
axs[1].set_ylabel("X")
axs[1].set_xticks([])

axs[2].plot(sol.t,sol.y[1,:],color=color_dict["Y"])
axs[2].axvline(156,linestyle="--",color="k",alpha=0.5)
axs[2].axvline(264.2+50,linestyle="--",color="k",alpha=0.5)
axs[2].set_ylabel("Y")
axs[2].set_xticks([])

axs[3].plot(sol.t,sol.y[2,:],color=color_dict["Z"])
axs[3].axvline(156,linestyle="--",color="k",alpha=0.5)
axs[3].axvline(264.2+50,linestyle="--",color="k",alpha=0.5)
axs[3].set_ylabel("Z")
axs[3].set_xticklabels(axs[3].get_xticks()[::-1])
axs[3].set_xlabel("time BP [10 kyr]")

plt.tight_layout()
#plt.savefig(f"./SM90_Output_t_end={t_cutoff}.pdf")

xs = np.array(sol.y[0,:])
ys = np.array(sol.y[1,:])
zs = np.array(sol.y[2,:])

In [ ]:
# altstep random walk
# idea here is to corrupt the time indices of the SM90 solution with a stochastic random walk
# so a psuedo-time index series is created with a variable time step
# in theory this should mimic the time corurption in ice core data

t_corrupted = np.zeros(int(t_cutoff/dt)) 
for i in range(1,len(t_corrupted)):
    # random walk using random step size drawn with gaussian w mean dt and sigma = 0.1*dt
    step = np.random.normal(loc = dt, scale = 0.1*dt)
    t_corrupted[i] = t_corrupted[i-1] + step

print(t_corrupted.size)
print(t_corrupted[-1])
print(sol.t.size)
print(sol.t[-1])

In [ ]:
# altstep

switchtime = 5
for i in range(int( (t_cutoff) / (switchtime) )):
    if i != 0:
        print(i)
        print(sol.t[i*5], " 10kyr")
        print(sol.t[i*5-1], " 10kyr")

        xs[i*50]=xs[i*5-1]
        ys[i*50]=ys[i*5-1]
        zs[i*50]=zs[i*5-1]
#print(xs)

plt.plot(sol.t, xs, color = 'black')
plt.plot(sol.t,sol.y[0,:],color=color_dict["X"])
plt.ylabel("X")
plt.xticks([])

In [ ]:
chunk_size = 500
best_chunk = 0
dist = 1e6

# calculates a the absolute and jacobian difference (i.e. absolute and slope discontinuity) between the start and end of a chunk 
for i in range(0, 4000-chunk_size):
    print(f" > Checking chunk starting at {i}...")
    chunkx = sol.y[0,i:i+chunk_size]
    chunky = sol.y[1,i:i+chunk_size]
    chunkz = sol.y[2,i:i+chunk_size]
    #chunkins = R_forcing(sol.t[i:i+chunk_size])

    chunkstart = np.array([chunkx[0],chunky[0],chunkz[0]])#,chunkins[0]])
    chunkend = np.array([chunkx[-1],chunky[-1],chunkz[-1]])#,chunkins[-1]])

    chunkdiff = np.linalg.norm(chunkend - chunkstart)
    chunkslopediff = np.linalg.norm(
        np.array([chunkx[-2]-chunkx[-1],chunky[-2]-chunky[-1],chunkz[-2]-chunkz[-1]])-#,chunkins[-2]-chunkins[-1]]) - 
        np.array([chunkx[1]-chunkx[0],chunky[1]-chunky[0],chunkz[1]-chunkz[0]])#,chunkins[1]-chunkins[0]])
    )

    newdist = chunkdiff + chunkslopediff

    if dist > newdist:
        best_chunk = i
        print(f" > New best chunk found at {i} with distance {newdist} (diff = {chunkdiff}, slope discontinuity = {chunkslopediff})")
        dist = newdist



In [ ]:
age_name,R_name,X_name,Y_name,Z_name = "age_kyrBP","Insol","X","Y","Z"
chunk = False

if chunk:
    R_Df = pd.DataFrame({age_name:np.round(sol.t[i:i+chunk_size]*1e1,1),R_name:R_Scaler.inverse_transform(R_forcing(sol.t[i:i+chunk_size]).reshape(-1, 1)).flatten()})
    X_Df = pd.DataFrame({age_name:np.round(sol.t[i:i+chunk_size]*1e1,1),X_name:sol.y[0,i:i+chunk_size]})
    Y_Df = pd.DataFrame({age_name:np.round(sol.t[i:i+chunk_size]*1e1,1),Y_name:sol.y[1,i:i+chunk_size]})
    Z_Df = pd.DataFrame({age_name:np.round(sol.t[i:i+chunk_size]*1e1,1),Z_name:sol.y[2,i:i+chunk_size]})
else:
    R_Df = pd.DataFrame({age_name:np.round(sol.t*1e1,1),R_name:R_Scaler.inverse_transform(R_forcing(sol.t).reshape(-1, 1)).flatten()})
    X_Df = pd.DataFrame({age_name:np.round(sol.t*1e1,1),'corrupted_age':t_corrupted*1e1,X_name:xs})
    Y_Df = pd.DataFrame({age_name:np.round(sol.t*1e1,1),'corrupted_age':t_corrupted*1e1,Y_name:ys})
    Z_Df = pd.DataFrame({age_name:np.round(sol.t*1e1,1),'corrupted_age':t_corrupted*1e1,Z_name:zs})

file_path = "Data/SM90"
#file_path = "Data/SM90_altstep"
R_Df.to_csv(f"{file_path}/4000kyr_SM90_Insol.csv")
X_Df.to_csv(f"{file_path}/4000kyr_SM90_X.csv")
Y_Df.to_csv(f"{file_path}/4000kyr_SM90_Y.csv")
Z_Df.to_csv(f"{file_path}/4000kyr_SM90_Z.csv")

In [ ]:
test = pd.read_csv(f"{file_path}/4000kyr_SM90_X.csv")
print(test)